In [ ]:
# %%
# Dog heart X-ray classification with a custom PyTorch CNN.
#
# Paste each "# %%" section into a Jupyter Notebook cell, or open this file in
# VS Code/Jupyter as a notebook-style Python file.
#
# If PyTorch is not installed, run one of these in a notebook cell first:
# CPU:
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# NVIDIA GPU, CUDA 12.1:
# %pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

# %%
from pathlib import Path
import csv
import random
import time

import matplotlib.pyplot as plt
from PIL import ImageFile, ImageOps

import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import datasets, transforms

ImageFile.LOAD_TRUNCATED_IMAGES = True


# %%
# Reproducibility and paths.
SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DATA_ROOT = Path("Dog_Heart") / "Dog_Heart"
if not DATA_ROOT.exists():
    DATA_ROOT = Path(r"D:\shiyanshi\deeplearning\Homework\Dog_Heart")

TRAIN_DIR = DATA_ROOT / "Train"
VALID_DIR = DATA_ROOT / "Valid"
TEST_DIR = DATA_ROOT / "Test" / "Images"

assert TRAIN_DIR.exists(), f"Missing train folder: {TRAIN_DIR}"
assert VALID_DIR.exists(), f"Missing valid folder: {VALID_DIR}"

OUTPUT_DIR = Path("outputs_dog_heart_cnn")
OUTPUT_DIR.mkdir(exist_ok=True)
BEST_MODEL_PATH = OUTPUT_DIR / "best_custom_cnn_dog_heart.pth"
PRED_CSV_PATH = OUTPUT_DIR / "Dog_Heart_text.csv"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Data root:", DATA_ROOT.resolve())


# %%
# Hyperparameters.
IMG_SIZE = 256

BATCH_SIZE = 16 if device.type == "cuda" else 8

EPOCHS = 50
PATIENCE = 15
MAX_LR = 1e-4

WEIGHT_DECAY = 5e-4

NUM_WORKERS = 0  # Safer for Windows/Jupyter. Increase to 2 or 4 if it works.
USE_WEIGHTED_SAMPLER = True
USE_WEIGHTED_LOSS = False


# %%
# Compute channel mean/std from the training set after resizing.
stats_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Lambda(lambda img: ImageOps.autocontrast(img)),
    transforms.ToTensor(),
])

stats_ds = datasets.ImageFolder(TRAIN_DIR, transform=stats_tfms)
stats_loader = DataLoader(
    stats_ds,
    batch_size=64,
    shuffle=False,
    num_workers=NUM_WORKERS,
)

channel_sum = torch.zeros(3)
channel_sq_sum = torch.zeros(3)
num_pixels = 0

for images, _ in stats_loader:
    channel_sum += images.sum(dim=(0, 2, 3))
    channel_sq_sum += (images ** 2).sum(dim=(0, 2, 3))
    num_pixels += images.size(0) * images.size(2) * images.size(3)

mean = channel_sum / num_pixels
std = (channel_sq_sum / num_pixels - mean ** 2).sqrt().clamp_min(1e-6)

mean = mean.tolist()
std = std.tolist()
print("Classes:", stats_ds.classes)
print("class_to_idx:", stats_ds.class_to_idx)
print("mean:", [round(x, 4) for x in mean])
print("std :", [round(x, 4) for x in std])


# %%
# Data augmentation and loaders.
train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.82, 1.0), ratio=(0.90, 1.10)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=7, fill=0),
    transforms.RandomAffine(degrees=0, translate=(0.03, 0.03), scale=(0.95, 1.05), fill=0),
    transforms.Lambda(lambda img: ImageOps.autocontrast(img)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.06), ratio=(0.3, 3.3), value="random"),
])

valid_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
    transforms.CenterCrop(IMG_SIZE),
    transforms.Lambda(lambda img: ImageOps.autocontrast(img)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std),
])

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=train_tfms)
valid_ds = datasets.ImageFolder(VALID_DIR, transform=valid_tfms)

class_names = train_ds.classes
num_classes = len(class_names)
targets = torch.tensor([label for _, label in train_ds.samples], dtype=torch.long)
class_counts = torch.bincount(targets, minlength=num_classes)
class_weights = class_counts.sum() / (num_classes * class_counts.float())

print("Train size:", len(train_ds), "Valid size:", len(valid_ds))
for idx, name in enumerate(class_names):
    print(f"{idx}: {name:6s} train={class_counts[idx].item():4d} weight={class_weights[idx].item():.3f}")

if USE_WEIGHTED_SAMPLER:
    sample_weights = [class_weights[label].item() for label in targets]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True,
    )
    train_shuffle = False
else:
    sampler = None
    train_shuffle = True

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=train_shuffle,
    sampler=sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

valid_loader = DataLoader(
    valid_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)


# %%
# Show a few augmented training images.
def denormalize(tensor, mean_values, std_values):
    x = tensor.detach().cpu().clone()
    for c, (m, s) in enumerate(zip(mean_values, std_values)):
        x[c] = x[c] * s + m
    return x.clamp(0, 1)


images, labels = next(iter(train_loader))
plt.figure(figsize=(10, 6))
for i in range(min(8, images.size(0))):
    plt.subplot(2, 4, i + 1)
    img = denormalize(images[i], mean, std).permute(1, 2, 0)
    plt.imshow(img)
    plt.title(class_names[labels[i].item()])
    plt.axis("off")
plt.tight_layout()
plt.show()


# %%
class SEBlock(nn.Module):
    """Squeeze-and-Excitation 模块"""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Linear(channels // reduction, channels, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        y = x.view(b, c, -1).mean(dim=2)  # Global Average Pooling
        y = self.fc1(y)
        y = self.relu(y)
        y = self.fc2(y)
        y = self.sigmoid(y).view(b, c, 1, 1)
        return x * y.expand_as(x)

class CustomConvBlock(nn.Module):
    """自定义序列块: Conv -> BatchNorm -> SiLU -> SE Block -> MaxPool"""
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.silu = nn.SiLU(inplace=True)
        self.se = SEBlock(out_channels)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.silu(x)
        x = self.se(x)
        x = self.pool(x)
        return x

class CustomCNN(nn.Module):
    """你的专属定制 CNN 网络"""
    def __init__(self, num_classes):
        super().__init__()
        # 特征提取层
        self.features = nn.Sequential(
            CustomConvBlock(3, 32),    # 输出: 128x128
            CustomConvBlock(32, 64),   # 输出: 64x64
            CustomConvBlock(64, 128),  # 输出: 32x32
            CustomConvBlock(128, 256), # 输出: 16x16
            CustomConvBlock(256, 512), # 输出: 8x8
        )
        # 分类层
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.SiLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# 实例化你的自定义模型
model = CustomCNN(num_classes=num_classes)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("✅ 成功加载：你专属定制的 Custom CNN 模型 (从头训练，无预训练权重)")
print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


# %%
# Training and evaluation utilities.
loss_weights = class_weights.to(device) if USE_WEIGHTED_LOSS else None
criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=0.03)
optimizer = torch.optim.AdamW(model.parameters(), lr=MAX_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=MAX_LR,
    epochs=EPOCHS,
    steps_per_epoch=len(train_loader),
    pct_start=0.20,
    div_factor=25,
    final_div_factor=100,
)

use_amp = device.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(images)
            loss = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_seen += batch_size

    return total_loss / total_seen, total_correct / total_seen


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_seen = 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        loss = criterion(logits, labels)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_seen += batch_size

    return total_loss / total_seen, total_correct / total_seen


# %%
# Train. The best validation-accuracy checkpoint is saved automatically.
history = {
    "train_loss": [],
    "train_acc": [],
    "valid_loss": [],
    "valid_acc": [],
}

best_acc = 0.0
epochs_without_improvement = 0
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    valid_loss, valid_acc = evaluate(model, valid_loader)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["valid_loss"].append(valid_loss)
    history["valid_acc"].append(valid_acc)

    improved = valid_acc > best_acc
    if improved:
        best_acc = valid_acc
        epochs_without_improvement = 0
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "class_names": class_names,
                "class_to_idx": train_ds.class_to_idx,
                "mean": mean,
                "std": std,
                "img_size": IMG_SIZE,
                "best_acc": best_acc,
                "epoch": epoch,
            },
            BEST_MODEL_PATH,
        )
    else:
        epochs_without_improvement += 1

    mark = "*" if improved else " "
    print(
        f"{mark} Epoch {epoch:03d}/{EPOCHS} | "
        f"train loss {train_loss:.4f} acc {train_acc*100:6.2f}% | "
        f"valid loss {valid_loss:.4f} acc {valid_acc*100:6.2f}% | "
        f"best {best_acc*100:6.2f}%"
    )

    if best_acc >= 0.80 and epochs_without_improvement >= 5:
        print("Validation accuracy is above 80%; stopping after stable improvement window.")
        break

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping: no validation improvement.")
        break

elapsed_min = (time.time() - start_time) / 60
print(f"Finished in {elapsed_min:.1f} min. Best valid accuracy: {best_acc*100:.2f}%")
print("Best checkpoint:", BEST_MODEL_PATH.resolve())


# %%
# Plot training curves.
epochs_range = range(1, len(history["train_acc"]) + 1)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["train_loss"], label="train")
plt.plot(epochs_range, history["valid_loss"], label="valid")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, [x * 100 for x in history["train_acc"]], label="train")
plt.plot(epochs_range, [x * 100 for x in history["valid_acc"]], label="valid")
plt.axhline(80, color="red", linestyle="--", linewidth=1, label="80%")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()

plt.tight_layout()
plt.show()


# %%
# Load best checkpoint and compute a confusion matrix on the validation set.
checkpoint = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

confusion = torch.zeros(num_classes, num_classes, dtype=torch.long)
valid_correct = 0
valid_total = 0

with torch.no_grad():
    for images, labels in valid_loader:
        images = images.to(device)
        labels = labels.to(device)
        preds = model(images).argmax(dim=1)
        valid_correct += (preds == labels).sum().item()
        valid_total += labels.numel()
        for true_label, pred_label in zip(labels.cpu(), preds.cpu()):
            confusion[true_label, pred_label] += 1

print(f"Best validation accuracy: {valid_correct / valid_total * 100:.2f}%")
print("Rows=true labels, columns=predicted labels")
print(confusion)

plt.figure(figsize=(5, 4))
plt.imshow(confusion.numpy(), cmap="Blues")
plt.xticks(range(num_classes), class_names, rotation=45)
plt.yticks(range(num_classes), class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.colorbar()

for i in range(num_classes):
    for j in range(num_classes):
        plt.text(j, i, str(confusion[i, j].item()), ha="center", va="center")

plt.tight_layout()
plt.show()


# %%
# Predict the unlabeled test images and save a CSV with the same two-column,
# no-header format as sample_results.csv: filename,label.
class TestImageDataset(Dataset):
    def __init__(self, image_dir, transform, ordered_names=None):
        self.image_dir = Path(image_dir)
        self.transform = transform

        if ordered_names is not None:
            self.paths = [self.image_dir / name for name in ordered_names]
            self.paths = [p for p in self.paths if p.exists()]
        else:
            suffixes = {".png", ".jpg", ".jpeg", ".bmp"}
            self.paths = sorted(
                [p for p in self.image_dir.iterdir() if p.suffix.lower() in suffixes],
                key=lambda p: (0, int(p.stem)) if p.stem.isdigit() else (1, p.name),
            )

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        path = self.paths[index]
        image = datasets.folder.default_loader(path)
        image = self.transform(image)
        return image, path.name


sample_csv = Path("sample_results.csv")
ordered_names = None
if sample_csv.exists():
    with sample_csv.open("r", newline="", encoding="utf-8") as f:
        ordered_names = [row[0] for row in csv.reader(f) if row]

test_ds = TestImageDataset(TEST_DIR, transform=valid_tfms, ordered_names=ordered_names)
test_loader = DataLoader(
    test_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)

rows = []
model.eval()
with torch.no_grad():
    for images, names in test_loader:
        images = images.to(device)
        preds = model(images).argmax(dim=1).cpu().tolist()
        rows.extend(zip(names, preds))

with PRED_CSV_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print("Saved predictions:", PRED_CSV_PATH.resolve())
print("Rows:", len(rows))
print("Label mapping:", train_ds.class_to_idx)
print(rows[:10])


# %%
# Optional: visualize test predictions.
preview_count = min(12, len(test_ds))
plt.figure(figsize=(12, 8))
for i in range(preview_count):
    image, name = test_ds[i]
    pred_label = dict(rows)[name]
    plt.subplot(3, 4, i + 1)
    img = denormalize(image, mean, std).permute(1, 2, 0)
    plt.imshow(img)
    plt.title(f"{name}\n{class_names[pred_label]}")
    plt.axis("off")
plt.tight_layout()
plt.show()